# 🏦 Exercícios SQL — CVM Data Lake
### Versão do Aluno | Big Data for Finance — FAE
**Banco:** PostgreSQL (DBeaver local)

**Tabelas utilizadas:**
- `layer_02_silver` — Tabelas que estão na camada Silver

**CNPJs de referência rápida:**
| Empresa | CNPJ |
|---|---|
| Petrobras | `33.000.167/0001-01` |
| Ambev | `07.526.557/0001-00` |
| Vale | `33.592.510/0001-54` |

---

## ⚙️ Configuração da Conexão

In [2]:
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine
import os

# 1. Carrega o arquivo .env (2 níveis acima se estiver em notebooks/01_bronze/)
load_dotenv('../../.env')

# 2. Lê as variáveis de ambiente
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# 3. Encoding de segurança: transforma 'senha@123' em 'senha%40123'
encoded_pass = quote_plus(DB_PASS)

# 4. Monta e cria a engine SQLAlchemy
conn_string = f"postgresql+psycopg2://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

---
## Questão 1 — Primeira olhada na tabela Silver 🟢 Fácil
**Contexto:** A primeira coisa que um analista faz ao chegar em um novo banco é explorar o que existe dentro da tabela. A camada Silver já tem os dados do Balanço Patrimonial limpos, normalizados e prontos para análise.

**Pergunta:** Mostre as **10 primeiras linhas** da tabela `n1_dfp_cia_aberta_bp`, exibindo apenas as colunas: `CNPJ_CIA`, `DENOM_CIA`, `DT_REFER`, `CD_CONTA`, `DS_CONTA`, `VL_CONTA_TRATADO` e `IS_LEAF`.

In [3]:
import pandas as pd

df = pd.read_sql(
    con=engine,
    sql="""  
select   
	"CNPJ_CIA", 
	"DENOM_CIA", 
	"DT_REFER", 
	"CD_CONTA", 
	"DS_CONTA",
	"VL_CONTA_TRATADO",
	"IS_LEAF"
from 
	layer_02_silver.n1_dfp_cia_aberta_bp
limit 10
"""
)

print(df)


             CNPJ_CIA                               DENOM_CIA   DT_REFER  \
0  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2013-12-31   
1  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2013-12-31   
2  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2013-12-31   
3  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2013-12-31   
4  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2013-12-31   
5  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2014-12-31   
6  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2014-12-31   
7  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2014-12-31   
8  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2014-12-31   
9  00.070.698/0001-11  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB 2014-12-31   

     CD_CONTA                                       DS_CONTA  \
0     2.03.02                            RESERVAS DE CAPITAL   
1  2.03.02.06    ADIANTAMENTO PARA 

---
## Questão 2 — Volume e cobertura da base 🟢 Fácil
**Contexto:** Antes de qualquer análise, o analista precisa entender o tamanho e a cobertura da base: quantas linhas existem, quantas empresas distintas estão cobertas e qual o intervalo de datas disponível.

**Pergunta:** Em uma única query, calcule para a tabela `n1_dfp_cia_aberta_bp`:
- Total de linhas → `total_linhas`
- Quantidade de empresas distintas (por `CNPJ_CIA`) → `empresas_distintas`
- Quantidade de datas de referência distintas (`DT_REFER`) → `datas_distintas`

In [4]:
df_2 = pd.read_sql(
    con=engine,
    sql="""  
select 
	count(*) as total_linhas,
	count(distinct "CNPJ_CIA") as empresas_distintas,
	count(distinct "DT_REFER") as datas_distintas
from 
	layer_02_silver.n1_dfp_cia_aberta_bp
"""
)

print(df_2)


   total_linhas  empresas_distintas  datas_distintas
0        180461                 185               54


---
## Questão 3 — Distribuição de empresas por setor 🟢 Fácil
**Contexto:** A CVM classifica cada empresa listada em um setor de atividade (`SETOR_ATIV`). Entender essa distribuição é o ponto de partida para qualquer análise setorial comparativa. Use a tabela `n0_empresas_selecionadas`, que contém o cadastro completo das empresas do nosso universo.

**Pergunta:** Quantas empresas existem em cada `SETOR_ATIV`? Ordene do setor com **mais** empresas para o com **menos**.

In [6]:
df_3 = pd.read_sql(
    con=engine,
    sql="""  
select
	"SETOR_ATIV",
	COUNT(*) as total_empresas
from 
	layer_02_silver.n0_empresas_selecionadas
group by "SETOR_ATIV"
order by count(*) desc;
"""
)

print(df_3)

                                    SETOR_ATIV  total_empresas
0                             Energia Elétrica              99
1   Construção Civil, Mat. Constr. e Decoração              90
2                  Comércio (Atacado e Varejo)              72
3              Serviços Transporte e Logística              51
4                      Metalurgia e Siderurgia              39
5                           Têxtil e Vestuário              39
6     Máquinas, Equipamentos, Veículos e Peças              39
7                                    Alimentos              27
8                 Saneamento, Serv. Água e Gás              24
9                             Serviços Médicos              24
10         Agricultura (Açúcar, Álcool e Cana)              21
11                   Comunicação e Informática              21
12                      Farmacêutico e Higiene              15
13                    Petroquímicos e Borracha              15
14                          Brinquedos e Lazer         

---
## Questão 4 — Petrobras: Ativo e Passivo mais recentes 🟢 Fácil
**Contexto:** No Balanço Patrimonial da CVM, `CD_CONTA = '1'` é o **Ativo Total** e `CD_CONTA = '2'` é o **Passivo + Patrimônio Líquido Total**. A equação fundamental do BP é: **Ativo = Passivo + PL** — qualquer diferença indica erro de dados.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`), mostre o valor (em bilhões de reais) das contas `'1'` e `'2'` para a **data de referência mais recente** disponível. Use uma subconsulta escalar com `MAX("DT_REFER")` para encontrar a data mais recente.

*Dica: `ROUND((valor / 1e9)::NUMERIC, 2)` converte para bilhões com 2 casas.*

In [ ]:
df_4 = pd.read_sql(
    con=engine,
    sql="""  
SELECT 
    "DENOM_CIA",
    "DT_REFER",
    "CD_CONTA",
    "DS_CONTA",
    ROUND(("VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2)  AS valor_bilhoes
FROM 
    layer_02_silver.n1_dfp_cia_aberta_bp
WHERE 
    "CNPJ_CIA" = '33.000.167/0001-01'
    AND "CD_CONTA" IN ('1', '2')
    AND "DT_REFER" = (
        SELECT MAX("DT_REFER") 
        FROM layer_02_silver.n1_dfp_cia_aberta_bp 
        WHERE "CNPJ_CIA" = '33.000.167/0001-01'
    );
"""
)

display(df_4)

,DENOM_CIA,DT_REFER,CD_CONTA,DS_CONTA,valor_bilhoes
0,PETROLEO BRASILEIRO S.A. PETROBRAS,2024-12-31,1,ATIVO TOTAL,1124.8
1,PETROLEO BRASILEIRO S.A. PETROBRAS,2024-12-31,2,PASSIVO TOTAL,1124.8


---
## Questão 5 — IS_LEAF: folhas vs. agregadores 🟢 Fácil
**Contexto:** `IS_LEAF = TRUE` marca contas **analíticas** (sem filhos) — são as únicas que devem ser somadas em qualquer ferramenta de BI, pois evitam dupla contagem. `IS_LEAF = FALSE` são contas **sintéticas** (pais) que já incorporam os valores de todas as suas contas filhas. Somar tudo geraria dupla contagem.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`) em `DT_REFER = '2023-12-31'`, quantas contas são `IS_LEAF = TRUE` e quantas são `IS_LEAF = FALSE`? Use `GROUP BY "IS_LEAF"`.

In [9]:
df_5 = pd.read_sql(
    con=engine,
    sql="""  
select
	"IS_LEAF",
	count(*)
from 
	layer_02_silver.n1_dfp_cia_aberta_bp
where "CNPJ_CIA" = '33.000.167/0001-01' and "DT_REFER" = '2023-12-31'
group by "IS_LEAF"

"""
)
display(df_5)

,IS_LEAF,count
0,False,51
1,True,19


---
## Questão 6 — Top 10 maiores empresas por Ativo Total em 2023 🟡 Médio
**Contexto:** Rankings de tamanho de balanço são análises clássicas de mercado de capitais. O Ativo Total (`CD_CONTA = '1'`) resume o tamanho total de uma empresa — tudo que ela possui e tem direito a receber.

**Pergunta:** Liste as **10 empresas com maior Ativo Total** em `DT_REFER = '2023-12-31'`. Exiba `DENOM_CIA`, `SETOR_ATIV` e o valor em **bilhões** (2 casas decimais). Ordene do maior para o menor.

*Dica: `ROUND((expr)::NUMERIC, 2)` — o cast para `NUMERIC` é obrigatório no PostgreSQL quando a coluna é `DOUBLE PRECISION`.*

In [12]:
df_6 = pd.read_sql(
    con=engine,
    sql="""  
select
	"DENOM_CIA",
	"SETOR_ATIV",
	ROUND(sum("VL_CONTA_TRATADO")::numeric, 2) as total_ativo
from 
	layer_02_silver.n1_dfp_cia_aberta_bp
where 
	"DT_REFER" = '2023-12-31'
	and "CD_CONTA" = '1'
group by "DENOM_CIA", "SETOR_ATIV"
order by sum("VL_CONTA_TRATADO") desc
limit 10
"""
)
display(df_6)

,DENOM_CIA,SETOR_ATIV,total_ativo
0,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,3.152664e+12
1,VALE S.A.,Extração Mineral,1.367952e+12
2,JBS S.A.,Alimentos,6.183962e+11
3,SUZANO S.A.,Papel e Celulose,4.307791e+11
4,COSAN S.A.,"Agricultura (Açúcar, Álcool e Cana)",4.195961e+11
5,AMBEV S.A.,Bebidas e Fumo,3.979324e+11
6,MARFRIG GLOBAL FOODS S.A.,Alimentos,3.928639e+11
7,TELEFÔNICA BRASIL S.A,Telecomunicações,3.622139e+11
8,EQUATORIAL S.A.,Energia Elétrica,3.109304e+11
9,BRASKEM S.A.,Petroquímicos e Borracha,2.752230e+11


---
## Questão 7 — Evolução anual do Ativo Total da Ambev 🟡 Médio
**Contexto:** Acompanhar a evolução do balanço ao longo dos anos revela a trajetória de crescimento (ou retração) de uma empresa. `DT_REFER` é do tipo `TIMESTAMP` — use `EXTRACT(YEAR FROM ...)` para extrair o ano.

**Pergunta:** Para a **Ambev** (`CNPJ_CIA = '07.526.557/0001-00'`), mostre a evolução anual do **Ativo Total** (`CD_CONTA = '1'`). Extraia o ano de `DT_REFER`, exiba o valor em bilhões e ordene do ano mais antigo ao mais recente.

In [ ]:
df_7 = pd.read_sql(
    con=engine,
    sql="""  
SELECT
    EXTRACT(YEAR FROM "DT_REFER") AS ano,
    ROUND((SUM("VL_CONTA_TRATADO") / 1e9)::numeric, 2) AS ativo_total_bilhoes
FROM 
    layer_02_silver.n1_dfp_cia_aberta_bp
WHERE 
    "CNPJ_CIA" = '07.526.557/0001-00'                
    AND "CD_CONTA" = '1'                         -- Ativo Total
GROUP BY 
    1
ORDER BY 
    ano ASC;
"""
)
display(df_7)

,ano,ativo_total_bilhoes
0,2013.0,68.673333
1,2014.0,72.143333
2,2015.0,90.176667
3,2016.0,83.840000
4,2017.0,86.853333
5,2018.0,94.126667
6,2019.0,101.743333
7,2020.0,125.196667
8,2021.0,138.603333
9,2022.0,137.956667


---
## Questão 8 — Contas reconstruídas pelo pipeline 🟡 Médio
**Contexto:** `FLAG_RECONSTRUCAO = TRUE` indica que uma linha foi criada **sinteticamente** pelo pipeline — a empresa não reportou aquela conta e o pipeline reconstruiu o valor somando os filhos. Mais contas reconstruídas = mais "buracos" no dado original enviado à CVM.

**Pergunta:** Em **toda a base histórica**, quais são as **10 combinações de empresa e data** com mais contas reconstruídas no BP? Mostre `DT_REFER`, `DENOM_CIA` e o total de contas de cada empresa. Use `SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END)` para contar. Agrupe por `"DT_REFER"` e `"DENOM_CIA"`. Ordene da mais para a menos reconstruída.


In [16]:
df_8 = pd.read_sql(
    con=engine,
    sql="""  
SELECT
    "DT_REFER",
    "DENOM_CIA",
    SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END) AS total_contas_reconstruidas
FROM 
    layer_02_silver.n1_dfp_cia_aberta_bp
GROUP BY 
    "DT_REFER", 
    "DENOM_CIA"
ORDER BY 
    total_contas_reconstruidas DESC
LIMIT 10;
"""
)
display(df_8)

,DT_REFER,DENOM_CIA,total_contas_reconstruidas
0,2022-12-31,CESP - CIA ENERGETICA DE SAO PAULO,15
1,2021-12-31,VIVER INCORPORADORA E CONSTRUTORA S.A.,2
2,2017-12-31,CIA SIDERURGICA NACIONAL,1
3,2010-12-31,USINAS SID DE MINAS GERAIS S.A.-USIMINAS,1
4,2016-12-31,CIA SIDERURGICA NACIONAL,1
5,2021-12-31,INTERNATIONAL MEAL COMPANY ALIMENTACAO S.A.,1
6,2022-12-31,GRUPO CASAS BAHIA S.A.,1
7,2022-12-31,CIA BRASILEIRA DE DISTRIBUICAO,1
8,2021-12-31,CVC BRASIL OPERADORA E AGÊNCIA DE VIAGENS S.A.,1
9,2018-12-31,PRIO S.A.,0


---
## Questão 9 — Base analítica de Receita Bruta por empresa via JOIN 🟡 Médio
**Contexto:** Antes de agregar, o analista precisa da base analítica: uma linha por empresa com seu setor e receita. Com esse DataFrame em mãos, o Pandas consegue gerar estatísticas descritivas por grupo com `.describe()`.

**Pergunta:** Faça um `JOIN` entre `n1_dfp_cia_aberta_dre` e `n0_empresas_selecionadas`. Para `DT_REFER = '2023-12-31'` e `CD_CONTA = '3.01'` (Receita Bruta), traga uma linha por empresa com `CNPJ_CIA`, `DENOM_CIA`, `SETOR_ATIV` e o valor em bilhões. Depois use `.groupby` + `.describe()` para ver as estatísticas por setor.


In [ ]:
df_9 = pd.read_sql(
    con=engine,
    sql="""  
SELECT
    t1."CNPJ_CIA",
    t1."DENOM_CIA",
    t2."SETOR_ATIV",
    ROUND((t1."VL_CONTA_TRATADO" / 1e9)::numeric, 2) AS receita_bilhoes
FROM 
    layer_02_silver.n1_dfp_cia_aberta_dre t1
INNER JOIN 
    layer_02_silver.n0_empresas_selecionadas t2 
    ON t1."CNPJ_CIA" = t2."CNPJ_CIA"
WHERE 
    t1."DT_REFER" = '2023-12-31'
    AND t1."CD_CONTA" = '3.01'
"""
)
display(df_9)

estatisticas_setor = df_9.groupby('SETOR_ATIV')['receita_bilhoes'].describe()

print(estatisticas_setor)

,CNPJ_CIA,DENOM_CIA,SETOR_ATIV,receita_bilhoes
0,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,0.35
1,01.107.327/0001-20,BBM LOGÍSTICA S.A.,Serviços Transporte e Logística,1.61
2,01.545.826/0001-07,GAFISA S.A.,"Construção Civil, Mat. Constr. e Decoração",1.10
3,02.302.101/0001-42,EMAE - EMPRESA METROP.AGUAS ENERGIA S.A.,Energia Elétrica,0.60
4,02.351.144/0001-18,TEGMA GESTAO LOGISTICA S.A.,Serviços Transporte e Logística,1.58
...,...,...,...,...
166,92.754.738/0001-62,LOJAS RENNER S.A.,Comércio (Atacado e Varejo),13.65
167,92.781.335/0001-02,TAURUS ARMAS S.A.,"Máquinas, Equipamentos, Veículos e Peças",1.78
168,96.418.264/0218-02,LOJAS QUERO QUERO S.A.,Comércio (Atacado e Varejo),2.40
169,97.191.902/0001-94,CONSERVAS ODERICH S.A.,Alimentos,0.78


                                            count        mean         std  \
SETOR_ATIV                                                                  
Agricultura (Açúcar, Álcool e Cana)           3.0   16.256667   20.594296   
Alimentos                                     7.0   77.231429  134.756058   
Bebidas e Fumo                                1.0   79.740000         NaN   
Bolsas de Valores/Mercadorias e Futuros       1.0    9.920000         NaN   
Brinquedos e Lazer                            3.0    1.700000    2.216213   
Comunicação e Informática                     7.0    2.040000    1.612245   
Comércio (Atacado e Varejo)                  22.0   15.213636   34.877844   
Construção Civil, Mat. Constr. e Decoração   30.0    1.679000    2.049228   
Educação                                      3.0    2.620000    0.989596   
Energia Elétrica                             14.0   12.590000   13.094201   
Extração Mineral                              1.0  208.070000         NaN   